# Jørgensen 2009 — Identification of More Risks Can Lead to Increased Over-Optimism

In [1]:
import re, utils

BRIEF_DOCS = utils.load("project brief")
STORY_DOCS = utils.load("user stories")

def build_j2009_tasks(less_instruction, more_instruction, documents, framing="",
                      success_question="the probability of project success (defined as: "
                                       "less than 25% estimation error and a client satisfied "
                                       "with quality and usability)"):
    tasks = []
    for condition, risk_instruction in [("LESS", less_instruction), ("MORE", more_instruction)]:
        for doc_name, doc_text in documents:
            prompt = f"""
You are a software developer asked to estimate the effort required to develop the system below.
{framing}
Note: this risk analysis is input to your effort estimate only — it is not the only risk
management activity that will take place on this project.

As a first step, {risk_instruction}

Then estimate the most likely effort required to develop and test the system, and assess
{success_question}.

```requirements
{doc_text}
```

End your response with exactly:
RISKS: <comma-separated list>
EFFORT: <number> work-hours
SUCCESS: <number>%
"""
            for model in utils.MODELS:
                tasks.append((model, prompt,
                              {"condition": condition, "doc": doc_name}))
    return tasks

def merge_j2009(store, model, response, meta):
    condition, doc = meta["condition"], meta["doc"]
    effort  = re.search(r"EFFORT:\s*(\d+)",  response or "")
    success = re.search(r"SUCCESS:\s*(\d+)", response or "")
    risks   = re.search(r"RISKS:\s*(.+)",    response or "")
    # Fallback: try extract_numbers if direct parse fails
    if not effort:
        eline = next((l for l in (response or "").splitlines() if "EFFORT" in l), "")
        nums = utils.extract_numbers(eline)
        effort_val = int(nums[0]) if nums else None
    else:
        effort_val = int(effort.group(1))
    row = {
        "effort":    effort_val,
        "success":   int(success.group(1))          if success else None,
        "num_risks": len(risks.group(1).split(",")) if risks   else None,
    }
    store.setdefault(condition, {}).setdefault(model, {})[doc] = row
    if row["effort"] is None:
        utils.record_failure("jorgensen2009_failures.jsonl", model, response, meta)
        print(f"  FAIL {condition} | {model.split('/')[1]:20} | {doc}")
    else:
        print(f"  OK   {condition} | {model.split('/')[1]:20} | {doc} | effort={row['effort']}")


## Experiment A — 3 most important risks vs as many as possible

Brief documents. LESS: identify the 3 most important risks. MORE: identify as many as possible.

In [ ]:
less_a = "briefly describe the THREE MOST IMPORTANT risk factors for this project."
more_a = "identify as many important risk factors as you can for this project."

results_a = {"LESS": {}, "MORE": {}}
utils.fire_and_collect(build_j2009_tasks(less_a, more_a, BRIEF_DOCS),
                       results_a, merge_j2009, save_path="jorgensen2009_a_partial.json")


## Experiment B — 1 risk vs structured retrospective analysis

MORE group spends at least 5 minutes recalling similar past projects and what went wrong, then identifies all risk factors with support from a checklist of common risk categories.

In [ ]:
less_b = "briefly describe the SINGLE MOST IMPORTANT risk factor for this project."
more_b = ("spend at least 5 minutes thinking back on similar development tasks you have worked on "
          "and what went wrong when completing those tasks. Consider risks such as: unclear or changing "
          "requirements, technical problems, personnel issues, integration difficulties, testing challenges, "
          "and scope creep. Then identify all important risk factors you can for this project.")

results_b = {"LESS": {}, "MORE": {}}
utils.fire_and_collect(build_j2009_tasks(less_b, more_b, BRIEF_DOCS),
                       results_b, merge_j2009, save_path="jorgensen2009_b_partial.json")


## Experiment C — 1 risk vs risk table with probability and severity

Different specification from A and B (user story documents). MORE group records probability and severity for each risk in a structured table. Success question asks about overrun probability rather than success probability.

In [ ]:
less_c = "briefly describe the SINGLE MOST IMPORTANT risk factor for this project."
more_c = ("identify all important risk factors you can. For each risk, complete a table with: "
          "(1) description, (2) probability of occurrence (low / medium / high), "
          "(3) severity of impact (low / medium / high).")

results_c = {"LESS": {}, "MORE": {}}
utils.fire_and_collect(
    build_j2009_tasks(less_c, more_c, STORY_DOCS,
                      success_question="the probability of experiencing more than 25% effort overrun"),
    results_c, merge_j2009, save_path="jorgensen2009_c_partial.json")


## Experiment D — Colleague's project (new experiment)

Motivation: Jørgensen's original Experiment D tested the MORE→optimism effect when estimators have no personal stake, using a conference scenario. Here the same LESS/MORE structure is applied to software estimation with a colleague framing. If the effect persists, it is not driven by personal investment.

In [ ]:
less_d = "briefly describe the THREE MOST IMPORTANT risk factors for your colleague's project."
more_d = "identify as many important risk factors as you can for your colleague's project."
colleague_framing = ("A colleague has asked you to provide an independent effort estimate for their project. "
                     "You have no personal stake in the outcome — you are simply offering an outside perspective.")

results_d = {"LESS": {}, "MORE": {}}
utils.fire_and_collect(build_j2009_tasks(less_d, more_d, BRIEF_DOCS, framing=colleague_framing),
                       results_d, merge_j2009, save_path="jorgensen2009_d_partial.json")

utils.save("jorgensen2009_results.json",
           {"exp_a": results_a, "exp_b": results_b, "exp_c": results_c, "exp_d": results_d})
